# 사전준비

In [ ]:
# 1. cuda-toolkit 12.6 설치 완료(nvcc --version)
# 2. pyproject.toml에 있는 transformers, datasets 지우기 (uv remove transformers datasets)
# 3. pyroject.toml 파일 업데이트 
# ```
# [tool.uv.sources]
# torch = { index = "pytorch" }
# torchvision = { index = "pytorch" }
# unsloth = { git = "https://github.com/unslothai/unsloth.git" }
# unsloth-zoo = { git = "https://github.com/unslothai/unsloth-zoo.git" }

# [[tool.uv.index]]
# name = "pytorch"
# url = "https://download.pytorch.org/whl/cu126"
# explicit = true
# ```
# 4. unsloth 설치 (uv add unsloth unsloth-zoo)
# 5. triton 설치 (uv pip install https://github.com/woct0rdho/triton-windows/releases/download/v3.2.0-windows.post10/triton-3.2.0-cp312-cp312-win_amd64.whl --no-deps)

# 1. 모델 불러오기

## 1) 베이스 모델 불러오기

In [16]:
from unsloth import FastLanguageModel 
import torch 

max_seq_length = 1024    # 한 번에 처리할 최대 토큰 수 (클수록 VRAM 많이 사용)
dtype = None             # GPU에 맞게 자동 선택 (None 권장)
load_in_4bit = True      # 4bit 압축 로드 여부 (True = VRAM 절약)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

print(f"✅ 모델 로드 완료!")
print(f"   dtype : {next(model.parameters()).dtype}")
print(f"   device: {next(model.parameters()).device}")

==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4070 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ 모델 로드 완료!
   dtype : torch.bfloat16
   device: cuda:0


## 2) LoRA 어댑터 준비

In [17]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                      # LoRA 크기 (클수록 학습 많이, VRAM 많이 사용)
    target_modules = [           # 어댑터 붙일 레이어 선택
        "q_proj", "k_proj",      #   └─ 어텐션 레이어
        "v_proj", "o_proj",
        "gate_proj", "up_proj",  #   └─ MLP 레이어
        "down_proj",
    ],
    lora_alpha = 16,             # LoRA 학습 강도 (보통 r과 같게 설정)
    lora_dropout = 0,            # 뉴런 드롭 비율 (0 = Unsloth 최적화)
    bias = "none",               # 바이어스 학습 여부 ("none" = Unsloth 최적화)
    use_gradient_checkpointing = "unsloth",  # VRAM 절약 방식 ("unsloth" 권장)
    random_state = 3407,         # 재현성을 위한 랜덤 시드
    use_rslora = False,          # Rank Stabilized LoRA 사용 여부
    loftq_config = None,         # LoftQ 양자화 설정 (실습에서는 불필요)
)

print("✅ LoRA 어댑터 추가 완료!")

✅ LoRA 어댑터 추가 완료!


# 2. 데이터셋 불러오기

## 1) 데이터셋 로드

In [18]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="data/custom_dataset.jsonl",
)

print(f"데이터 수: {len(dataset)}")
print(f"컬럼: {dataset.column_names}")
print(dataset)

데이터 수: 1
컬럼: {'train': ['messages']}
DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 56
    })
})


## 2) 데이터 분할

In [26]:
split = dataset["train"].train_test_split(test_size=0.2, seed=42)
split

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 44
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 12
    })
})

In [27]:
train_dataset = split["train"]
valid_dataset = split["test"]

In [28]:
train_dataset[0]

{'messages': [{'role': 'system',
   'content': '당신은 온라인 쇼핑몰의 친절하고 신속한 고객 상담사입니다. 고객의 문의에 공감하며 정중하고 이해하기 쉽게 답변해주세요.'},
  {'role': 'user', 'content': '주문 후 결제 수단을 바꿀 수 있나요?'},
  {'role': 'assistant',
   'content': '결제 완료 후에는 결제 수단을 직접 변경하기 어려운 경우가 많아요 😊 보통은 기존 주문을 취소한 뒤 원하는 결제 수단으로 다시 주문해주셔야 합니다. 출고 전이라면 빠르게 처리해보시는 걸 추천드려요!'}]}

In [22]:
tokenizer.apply_chat_template(
    [
        {'role': 'system','content': '당신은 온라인 쇼핑몰의 친절하고 신속한 고객 상담사입니다. 고객의 문의에 공감하며 정중하고 이해하기 쉽게 답변해주세요.'},
        {'role': 'user', 'content': '주문 후 결제 수단을 바꿀 수 있나요?'},
        {'role': 'assistant','content': '결제 완료 후에는 결제 수단을 직접 변경하기 어려운 경우가 많아요 😊 보통은 기존 주문을 취소한 뒤 원하는 결제 수단으로 다시 주문해주셔야 합니다. 출고 전이라면 빠르게 처리해보시는 걸 추천드려요!'}
    ],
    tokenize=False,
    add_generation_prompt=False
)

'<|im_start|>system\n당신은 온라인 쇼핑몰의 친절하고 신속한 고객 상담사입니다. 고객의 문의에 공감하며 정중하고 이해하기 쉽게 답변해주세요.<|im_end|>\n<|im_start|>user\n주문 후 결제 수단을 바꿀 수 있나요?<|im_end|>\n<|im_start|>assistant\n결제 완료 후에는 결제 수단을 직접 변경하기 어려운 경우가 많아요 😊 보통은 기존 주문을 취소한 뒤 원하는 결제 수단으로 다시 주문해주셔야 합니다. 출고 전이라면 빠르게 처리해보시는 걸 추천드려요!<|im_end|>\n'

## 2) formatting_func 만들기

In [23]:
def formatting_prompts_func(data):
    result = []
    messages = data["messages"]  # 이미 하나의 샘플

    # messages가 딕셔너리 리스트인지 확인
    if isinstance(messages[0], dict):
        # 단일 샘플로 들어온 경우
        chat = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        result.append(chat)
    else:
        # 배치로 들어온 경우
        for msgs in messages:
            chat = tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=False
            )
            result.append(chat)
    return result

# 3. Trainer 만들기

## 1) TrainingArgument

In [29]:
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

args = TrainingArguments(
    per_device_train_batch_size = 2,         # GPU당 배치 크기
    gradient_accumulation_steps = 4,          # 실질 배치 = 2×4 = 8
    warmup_steps = 5,                         # lr 워밍업 스텝 수
    # max_steps = 100,                        # 테스트용 (전체 학습 시 num_train_epochs 사용)
    num_train_epochs = 10,                    # 전체 학습 시 이걸 사용
    learning_rate = 2e-4,                     # LoRA 권장 학습률
    fp16 = not is_bfloat16_supported(),       # 구형 GPU용
    bf16 = is_bfloat16_supported(),           # 최신 GPU용 (자동 감지)
    logging_steps = 1,                        # 몇 스텝마다 로그 출력할지
    optim = "adamw_8bit",                     # VRAM 절약형 옵티마이저
    weight_decay = 0.01,                      # 과적합 방지 정규화
    lr_scheduler_type = "linear",             # lr 감소 방식
    seed = 1234,                              # 재현성을 위한 랜덤 시드
    output_dir = "outputs",                   # 체크포인트 저장 폴더
    report_to = "none",                       # 로그 기록 위치 (wandb 등)
    average_tokens_across_devices = False,    # 
    load_best_model_at_end = True,            # ➕ 이거 없으면 EarlyStopping 에러남
    eval_strategy="epoch",                    # ➕ 추가
    save_strategy="epoch"                     # ➕ 추가
)

## 2) Early Stopping

In [30]:
from transformers import EarlyStoppingCallback

callbacks = [
    EarlyStoppingCallback(
        early_stopping_patience = 3   # 3번 연속 안 좋아지면 멈춤
    )
]

## 3) SFTTrainer

In [31]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = valid_dataset,
    formatting_func = formatting_prompts_func,      # 학습에 사용할 컬럼명
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,               # 전처리 CPU 코어 수
    packing = False,                    # 멀티턴 데이터는 False 권장
    args = args,
    callbacks=callbacks
)

print("✅ 트레이너 설정 완료!")

Unsloth: Tokenizing ["text"]:   0%|          | 0/44 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/12 [00:00<?, ? examples/s]

✅ 트레이너 설정 완료!


# 4. 학습

## 1) 학습 전 GPU 현황

In [32]:
# 학습 전 GPU 메모리 상태 기록 (학습 후 셀과 비교용)
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU 이름     : {gpu_stats.name}")
print(f"전체 VRAM    : {max_memory} GB")
print(f"현재 예약량  : {start_gpu_memory} GB")
print(f"남은 여유    : {round(max_memory - start_gpu_memory, 3)} GB")

GPU 이름     : NVIDIA GeForce RTX 4070 Laptop GPU
전체 VRAM    : 7.996 GB
현재 예약량  : 3.02 GB
남은 여유    : 4.976 GB


## 2) 학습하기

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44 | Num Epochs = 10 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Epoch,Training Loss,Validation Loss
1,2.058600,1.946658
2,1.358000,1.286548
3,1.071700,1.087540
4,0.775200,1.030417
5,0.804700,1.018508


In [ ]:
# 학습 후 GPU 메모리 및 시간 통계
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"✅ 학습 완료!")
print(f"")
print(f"⏱  학습 시간        : {round(trainer_stats.metrics['train_runtime'] / 60, 2)} 분")
print(f"")
print(f"💾 전체 VRAM 사용량 : {used_memory} GB ({used_percentage} %)")
print(f"💾 LoRA 학습 사용량 : {used_memory_for_lora} GB ({lora_percentage} %)")
print(f"   (모델 로드 제외, 순수 학습에 쓴 VRAM)")

✅ 학습 완료!

⏱  학습 시간        : 7.55 분

💾 전체 VRAM 사용량 : 3.18 GB (39.77 %)
💾 LoRA 학습 사용량 : 1.602 GB (20.035 %)
   (모델 로드 제외, 순수 학습에 쓴 VRAM)


## 3) 모델 저장

In [ ]:
# 학습 후 16bit로 병합 저장
model.save_pretrained_merged(
    "C:/potenup3/models/model_merged_qwen_custom",
    tokenizer,
    save_method="merged_16bit"
)

Found HuggingFace hub cache directory: C:\Users\user\.cache\huggingface\hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [03:53<00:00, 233.45s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:52<00:00, 112.89s/it]


Unsloth: Merge process complete. Saved to `g:\내 드라이브\에이치소프트\원티드랩\원티드랩3기\원티드랩_포텐업_AI 에이전트 트랙 3기\강의자료\3월 딥러닝 활용\실습 자료\models\model_merged_qwen_teddynote`


# 5. 추론

## 1) 모델 불러온 후 추론

In [ ]:
# unsloth Bug : 현재 FastLanguageModel로 추론하는 것은 버그 존재
# 이때 model에 unsloth가 반영되어 있기 때문에 재시작 한 후에 불러와야 한다.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# ── 병합된 모델 로드 ──────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    "C:/potenup3/models/model_merged_qwen_custom",
    dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("C:/potenup3/models/model_merged_qwen_custom")

print("✅ 모델 로드 완료!")

The tokenizer you are loading from 'models/model_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ 모델 로드 완료!


In [4]:
input_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "테디노트 유튜브 채널에 대해 알려주세요"}],
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
print(inputs)

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198, 130229,  89235, 127121,
          28626, 126310, 144293, 131196,   3315,    109,    226, 139287,  19391,
         131978, 138630,  91669, 151645,    198, 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}


In [ ]:
# AttributeError: 'Qwen2Attention' object has no attribute 'apply_qkv' -> 커널 재시작 후 모델 다시 불러와주세요
# model 속에 unsloth가 반영되어있어 이를 없애기 위해 재시작이 필요합니다.
from transformers import TextStreamer

print("=== 학습 후 출력 (After) ===")
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    use_cache=False,
    repetition_penalty=1.3,  
    temperature=0.7,      
    do_sample=True,           
)

=== 학습 후 출력 (After) ===
테디노트(TeddyNote)는 데이터 분석과 머신러닝 주제를 다루며, 강의와 튜토리얼을 제공하는 YouTube 플랫폼입니다. 이 범용 AI가 다양한 기업 및 교육기관에서 활용하기 위해 공개됩니다.<|im_end|>


# 2) Ollama에 내 모델 등록하기

## 1) Unsloth로 모델 불러오기

In [ ]:
from unsloth import FastLanguageModel 
import torch 

max_seq_length = 1024    # 한 번에 처리할 최대 토큰 수 (클수록 VRAM 많이 사용)
dtype = None             # GPU에 맞게 자동 선택 (None 권장)
load_in_4bit = False      # 4bit 압축 로드 여부 (True = VRAM 절약) ✅

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "C:/potenup3/models/model_merged_qwen_custom",
    max_seq_length = max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

print(f"✅ 모델 로드 완료!")

C:\Users\user\AppData\Local\Temp\ipykernel_39904\3943191581.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4070 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


[accelerate.big_modeling|WARNING]Some parameters are on the meta device because they were offloaded to the cpu.


✅ 모델 로드 완료!


## 2) GGUF로 모델 저장하기

In [ ]:
# GGUF 변환 + 저장
# cmake installer, visual studio installer, openssl가 자동으로 뜸
# https://cmake.org/download/
# https://visualstudio.microsoft.com/ko/downloads/
# https://openssl-library.org/source/
# vscode 껏다 켜기
model.save_pretrained_gguf(
    "C:/potenup3/models/model_merged_qwen_custom_gguf",           # 저장 폴더
    tokenizer,
    quantization_method = "q4_k_m"        # 양자화 방식
)

Unsloth: Model is not a PEFT model. Using existing checkpoint at models/model_merged
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['models/model_merged_gguf\\model_merged.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['models/model_merged_gguf\\model_merged.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'models/model_merged'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: C:\Users\user\.unsloth\llama.cpp\build\bin\Release\llama-cli.exe --model models/model_merged_gguf\model_merged.Q4_K_M.gguf -p "why is the sky blue?"


{'save_directory': 'models/model_merged',
 'gguf_directory': 'models/model_merged_gguf',
 'gguf_files': ['models/model_merged_gguf\\model_merged.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
# GGUF 변환 + 저장
# cmake installer, visual studio installer, openssl가 자동으로 뜸
# https://cmake.org/download/
# https://visualstudio.microsoft.com/ko/downloads/
# https://openssl-library.org/source/
# vscode 껏다 켜기
model.save_pretrained_gguf(
    "C:/potenup3/models/model_merged_qwen_custom_gguf",           # 저장 폴더
    tokenizer,
    quantization_method = "q8_0"        # 양자화 방식
)

## 3) Ollama에 내 모델 GGUF 등록하기

In [ ]:
# 터미널에서 실행하세요 
# cd models/model_merged_gguf
# ollama create 내모델이름설정 -f Modelfile 
# ollama Desktop 켜보세요